# Mini_Assignment_2_Varun_Gahlot

## Domestic Flight Delay Records

### Outline

1. Project Summary
2. Setup and Spark Session
3. Load Dataset
4. Task 1 – Count Flights That Arrived Earlier Than Expected
5. Task 2 – Typical Departure Time for Flights Over 2000 Miles
6. Task 3 – Proportion of Flights With Arrival Delays Longer Than 60 Minutes
7. Task 4 – Average Airtime for Flights Departing Before 9:00 AM
8. Task 5 – Maximum Arrival Delay for Flights With No Departure Delay
9. Assumptions
10. Final Results Summary

## 1. Project Summary

### Overview

This mini assignment analyzes the **Domestic Flight Delay Records** dataset using **PySpark**. The dataset contains information about domestic flights in the United States, including flight dates, departure and arrival delays, airtime, flight distance, and scheduled departure and arrival times.

The objective of this assignment is to use PySpark functions such as filtering, aggregation, counting, averaging, and finding maximum values to answer five questions related to flight delays and travel characteristics.

### Objectives

The assignment focuses on:

- Identifying the number of flights that arrived earlier than expected.
- Determining the typical departure time for flights traveling more than 2000 miles.
- Calculating the proportion of flights with arrival delays longer than 60 minutes.
- Calculating the average airtime for flights departing before 9:00 AM.
- Finding the maximum arrival delay among flights that experienced no departure delay.

Each task is implemented as a reusable PySpark function, and the resulting outputs are presented and interpreted.

#Installing PySpark

In [1]:
!pip install pyspark -q

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Mini_Assignment_2_Flight_Delay") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark session created successfully!")

Spark version: 4.0.4
Spark session created successfully!


#Importing Files

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Flight Dataset - CSV(in).csv to Flight Dataset - CSV(in).csv


In [3]:
df = spark.read.csv(
    "Flight Dataset - CSV(in).csv",
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully!")

Dataset loaded successfully!


#Task 1 – Count Flights That Arrived Earlier Than Expected

In [4]:
df.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- DEP_DELAY: integer (nullable = true)
 |-- ARR_DELAY: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)



In [5]:
df.show(5)

+--------+---------+---------+--------+--------+---------+---------+
| FL_DATE|DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE| DEP_TIME| ARR_TIME|
+--------+---------+---------+--------+--------+---------+---------+
|1/1/2006|        5|       19|     350|    2475| 9.083333|12.483334|
|1/2/2006|      167|      216|     343|    2475|11.783334|15.766666|
|1/3/2006|       -7|       -2|     344|    2475| 8.883333|12.133333|
|1/4/2006|       -5|      -13|     331|    2475| 8.916667|    11.95|
|1/5/2006|       -3|      -17|     321|    2475|     8.95|11.883333|
+--------+---------+---------+--------+--------+---------+---------+
only showing top 5 rows


In [6]:
def count_early_arrivals(df):
    early_arrivals = df.filter(df.ARR_DELAY < 0)
    count = early_arrivals.count()
    return count

In [7]:
early_arrival_count = count_early_arrivals(df)

print("Number of flights that arrived earlier than expected:", early_arrival_count)

Number of flights that arrived earlier than expected: 534655


#Task 2 – Typical Departure Time for Flights Over 2000 Miles

In [12]:
from pyspark.sql.functions import avg

In [13]:
def typical_departure_time(df):
    long_flights = df.filter(df.DISTANCE > 2000)
    typical_time = long_flights.agg(avg("DEP_TIME")).first()[0]
    return typical_time

In [14]:
result = typical_departure_time(df)

print("Typical departure time for flights over 2000 miles:", result)

Typical departure time for flights over 2000 miles: 13.973233947624726


#Task 3: proportion of flights with arrival delays longer than 60 minutes.

In [15]:
def delayed_over_60_proportion(df):
    delayed_flights = df.filter(df.ARR_DELAY > 60)
    delayed_count = delayed_flights.count()
    total_flights = df.count()

    proportion = delayed_count / total_flights

    return proportion

In [17]:
result = delayed_over_60_proportion(df)

print("Proportion of flights with arrival delays longer than 60 minutes:", result)

result * 100

Proportion of flights with arrival delays longer than 60 minutes: 0.053066


5.3066

#Task 4 - Create a function that gives the average airtime for flights that left earlier than 9:00 AM.

In [18]:
def average_airtime_before_9(df):
    early_flights = df.filter(df.DEP_TIME < 9)
    average_airtime = early_flights.agg(avg("AIR_TIME")).first()[0]
    return average_airtime

In [19]:
result = average_airtime_before_9(df)

print("Average airtime for flights departing before 9:00 AM:", result)

Average airtime for flights departing before 9:00 AM: 111.36120276990287


#Task 5 — Create a function that determines the maximum arrival delay for flights that did not experience a delay upon departure.

In [20]:
from pyspark.sql.functions import max

def max_arrival_delay_no_departure_delay(df):
    no_departure_delay = df.filter(df.DEP_DELAY == 0)
    maximum_arrival_delay = no_departure_delay.agg(max("ARR_DELAY")).first()[0]
    return maximum_arrival_delay

In [21]:
result = max_arrival_delay_no_departure_delay(df)

print("Maximum arrival delay for flights with no departure delay:", result)

Maximum arrival delay for flights with no departure delay: 232


#Assumptions

**Assumptions**

The following assumptions were made while completing the analysis:

1. A negative `ARR_DELAY` indicates that a flight arrived earlier than its expected arrival time. Therefore, flights with `ARR_DELAY < 0` are considered early arrivals.

2. For Task 2, flights with `DISTANCE > 2000` are considered flights traveling over 2000 miles.

3. The term "typical departure time" in Task 2 is interpreted as the arithmetic average of the `DEP_TIME` values for flights traveling over 2000 miles.

4. The `DEP_TIME` values are stored as decimal representations of time. Therefore, the calculated average is interpreted as a decimal-hour representation of the typical departure time.

5. For Task 3, an arrival delay is considered longer than 60 minutes only when `ARR_DELAY > 60`. A delay of exactly 60 minutes is not included.

6. The proportion in Task 3 is calculated as the number of flights with `ARR_DELAY > 60` divided by the total number of flights.

7. For Task 4, flights departing before 9:00 AM are identified using `DEP_TIME < 9`.

8. For Task 5, a flight is considered to have experienced no departure delay when `DEP_DELAY == 0`.

9. For Task 5, the maximum value of `ARR_DELAY` among the qualifying flights is returned.

10. Null or missing values in the relevant columns are handled by Spark's filtering and aggregation behavior and are not manually imputed.

11. The dataset is treated as provided for the purpose of this assignment, and no external data is introduced into the analysis.

## 10. Final Results Summary

## Final Results Summary

| Task | Result |
|------|--------|
| Task 1 – Early arrivals | 534,655 flights |
| Task 2 – Typical departure time for flights over 2000 miles | 13.9732 (~1:58 PM) |
| Task 3 – Proportion with arrival delay > 60 minutes | 0.053066 (~5.31%) |
| Task 4 – Average airtime before 9:00 AM | 111.3612 minutes |
| Task 5 – Maximum arrival delay with no departure delay | 232 minutes |

The analysis demonstrates the use of PySpark filtering, counting, aggregation, averaging, and maximum-value operations to analyze flight delay data.